# Create geographical base file

This script loads a bunch of official GIS data and creates a single file that shows:

- Which bidding zones a municipality belongs to and to which extent
- Which region a municipality belongs to
- Which federation a municipality belongs to

It contains both codes and clear text values.

In [1]:
# Import libraries and set basic variables

import sys
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

root_path = Path(globals()['_dh'][0]).resolve().parent.parent.parent
sys.path.append(str(root_path))

from paths import input_path
from generator.library.utilities import get_path

In [7]:
## This data has been taken from the source code of nätområden.se
## It has been cleaned up (delete a property with html code) and stored as a geojson file

biddingzones = gpd.read_file("biddingzones.geojson", encoding='utf-8')

## The data below is an unofficial version of Sweden's municipalities taken from Opendatasoft

municipalities = gpd.read_file("georef-sweden-kommun@public.geojson", encoding='utf-8')

## Reproject the two frames to get accurate areas
municipalities = municipalities.to_crs(epsg=3006)
biddingzones = biddingzones.to_crs(epsg=3006)

Skipping field center: unsupported OGR type: 3


In [15]:
# Calculate

# Perform spatial intersection to find overlapping areas
intersection = gpd.overlay(municipalities, biddingzones, how='intersection')

# Add a column for the area of the intersected geometry
intersection['intersected_area'] = intersection.geometry.area

# Add a column for the original municipality's total area
municipalities['total_area'] = municipalities.geometry.area

# Merge intersected GeoDataFrame with municipalities' total area using kom_code
intersection = intersection.merge(
    municipalities[['kom_code', 'total_area']],
    left_on='kom_code',
    right_on='kom_code',
    suffixes=('', '_muni')
)

# Calculate the percentage area of each municipality in each bidding zone
intersection['percentage_area'] = (intersection['intersected_area'] / 
                                    intersection['total_area'])


In [16]:
intersection.loc[intersection['kom_name'] == 'Arjeplog']

,geo_point_2d,year,lan_code,lan_name,kom_code,kom_name,kom_area_code,kom_type,total_area,omr,namn,bolag,snitt,id,geometry,intersected_area,total_area_muni,percentage_area
0,"{ ""lon"": 17.239017732460756, ""lat"": 66.3645005...",2022,25,Norrbottens län,2506,Arjeplog,SWE,Kommun,1.458153e+10,None,SE2,None,2,SE2,"POLYGON ((520428.91 7358424.673, 520441.02 735...",3.017875e+03,1.458153e+10,2.069657e-07
1,"{ ""lon"": 17.239017732460756, ""lat"": 66.3645005...",2022,25,Norrbottens län,2506,Arjeplog,SWE,Kommun,1.458153e+10,None,SE1,None,1,SE1,"POLYGON ((527543.431 7386296.07, 530038.181 73...",1.358218e+10,1.458153e+10,9.314648e-01


In [17]:
intersection[intersection['kom_code'].duplicated(keep=False)]

,geo_point_2d,year,lan_code,lan_name,kom_code,kom_name,kom_area_code,kom_type,total_area,omr,namn,bolag,snitt,id,geometry,intersected_area,total_area_muni,percentage_area
0,"{ ""lon"": 17.239017732460756, ""lat"": 66.3645005...",2022,25,Norrbottens län,2506,Arjeplog,SWE,Kommun,1.458153e+10,None,SE2,None,2,SE2,"POLYGON ((520428.91 7358424.673, 520441.02 735...",3.017875e+03,1.458153e+10,2.069657e-07
1,"{ ""lon"": 17.239017732460756, ""lat"": 66.3645005...",2022,25,Norrbottens län,2506,Arjeplog,SWE,Kommun,1.458153e+10,None,SE1,None,1,SE1,"POLYGON ((527543.431 7386296.07, 530038.181 73...",1.358218e+10,1.458153e+10,9.314648e-01
9,"{ ""lon"": 12.243643323264264, ""lat"": 57.1579433...",2022,13,Hallands län,1383,Varberg,SWE,Kommun,1.710687e+09,None,SE4,None,4,SE4,"POLYGON ((358504.171 6344993.838, 357916.61 63...",3.153454e+04,1.710687e+09,1.843385e-05
10,"{ ""lon"": 12.243643323264264, ""lat"": 57.1579433...",2022,13,Hallands län,1383,Varberg,SWE,Kommun,1.710687e+09,None,SE3,None,3,SE3,"POLYGON ((321581.693 6356717.101, 323347.144 6...",1.006077e+09,1.710687e+09,5.881128e-01
23,"{ ""lon"": 16.377957613599087, ""lat"": 61.3125715...",2022,21,Gävleborgs län,2183,Bollnäs,SWE,Kommun,1.987929e+09,None,SE2,None,2,SE2,"POLYGON ((555643.674 6780200.052, 556246.648 6...",1.788450e+09,1.987929e+09,8.996552e-01
24,"{ ""lon"": 16.377957613599087, ""lat"": 61.3125715...",2022,21,Gävleborgs län,2183,Bollnäs,SWE,Kommun,1.987929e+09,None,SE3,None,3,SE3,"POLYGON ((591100.968 6782061.786, 590835.244 6...",8.013409e+08,1.987929e+09,4.031034e-01
26,"{ ""lon"": 13.804976625081229, ""lat"": 57.3560733...",2022,06,Jönköpings län,0617,Gnosjö,SWE,Kommun,4.514935e+08,None,SE4,None,4,SE4,"MULTIPOLYGON (((430460.565 6347382.572, 430506...",3.359955e+07,4.514935e+08,7.441869e-02
27,"{ ""lon"": 13.804976625081229, ""lat"": 57.3560733...",2022,06,Jönköpings län,0617,Gnosjö,SWE,Kommun,4.514935e+08,None,SE3,None,3,SE3,"POLYGON ((415888.714 6362102.114, 416128.975 6...",5.244949e+07,4.514935e+08,1.161689e-01
39,"{ ""lon"": 15.801347438463512, ""lat"": 57.3986937...",2022,08,Kalmar län,0860,Hultsfred,SWE,Kommun,1.195258e+09,None,SE4,None,4,SE4,"POLYGON ((562929.923 6353022.814, 562838.287 6...",3.030428e+08,1.195258e+09,2.535375e-01
40,"{ ""lon"": 15.801347438463512, ""lat"": 57.3986937...",2022,08,Kalmar län,0860,Hultsfred,SWE,Kommun,1.195258e+09,None,SE3,None,3,SE3,"POLYGON ((531301.156 6361165.388, 531296.954 6...",6.799645e+08,1.195258e+09,5.688850e-01
